# Supplementary Figure — network localization and null analyses

This notebook is a figure-only companion to the strict network-localization
analysis. It does not recompute permutations.

It loads:
- the strict random-archetype null;
- the matched-ratio spatial/temporal summary.

The figure emphasizes null-calibrated localization metrics and matched-ratio
spatial-versus-temporal differences. Descriptive Yeo composition in the main
motif figure should not be interpreted as enrichment without these nulls.

### Path note\n\nThe original network-localization analysis saved its exported CSVs to an absolute legacy path under `outputs/notebook_figures/013_network_localization_analysis/`. This version searches that location plus common relative project paths.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NULL_NAME = "strict_random_archetype_null_across.csv"
MATCHED_NAME = "matched_ratio_spatial_temporal_summary_across.csv"

candidate_dirs = [
    Path("013_network_localization_analysis"),
    Path("figures/013_network_localization_analysis"),
    Path("../figures/013_network_localization_analysis"),
    Path("../../figures/013_network_localization_analysis"),
    Path("outputs/notebook_figures/013_network_localization_analysis"),
]

found_pairs = []

for d in candidate_dirs:
    null_path = d / NULL_NAME
    matched_path = d / MATCHED_NAME
    if null_path.exists() and matched_path.exists():
        found_pairs.append((null_path, matched_path))

if not found_pairs:
    for root in [Path.cwd(), Path.cwd().parent]:
        if not root.exists():
            continue
        try:
            for null_path in root.glob(f"**/{NULL_NAME}"):
                matched_path = null_path.parent / MATCHED_NAME
                if matched_path.exists():
                    found_pairs.append((null_path, matched_path))
                    break
        except (OSError, PermissionError):
            pass

        if found_pairs:
            break

if not found_pairs:
    print("Current working directory:", Path.cwd())
    print("\nChecked these likely directories:")
    for d in candidate_dirs:
        print("  ", d)

    raise FileNotFoundError(
        "\nCould not find the exported network-localization tables.\n\n"
        "Expected these two files in the SAME directory:\n"
        f"  {NULL_NAME}\n"
        f"  {MATCHED_NAME}\n\n"
        "The original analysis notebook saved them under:\n"
        "  outputs/notebook_figures/"
        "013_network_localization_analysis/\n\n"
        "If that folder no longer exists, rerun only the FINAL EXPORT CELL of "
        "013_network_localization_analysis_v5_strict_null_matched_ratio after "
        "changing FIG_ROOT to your current project directory. The expensive "
        "permutation analyses do not need to be rerun if those dataframes are "
        "still in memory."
    )

NULL_CSV, MATCHED_CSV = found_pairs[0]
DATA_DIR = NULL_CSV.parent

print("Using network-localization outputs from:")
print(" ", DATA_DIR)

null_df = pd.read_csv(NULL_CSV)
matched = pd.read_csv(MATCHED_CSV)

OUTPUT_DIR = Path("outputs/paper_supplement")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FIG_PNG = OUTPUT_DIR / "Supplement_network_localization_nulls.png"
FIG_PDF = OUTPUT_DIR / "Supplement_network_localization_nulls.pdf"

display(null_df.head())
display(matched.head())

In [ ]:
# ============================================================
# SUPPLEMENTARY FIGURE — NETWORK LOCALIZATION NULLS
#
# A/B only:
#   A = Gini localization relative to random archetypes
#   B = participation ratio relative to random archetypes
#
# Encoding:
#   purple = intact
#   green  = word-scrambled
#   solid  = spatial MS-AA
#   dashed = temporal MS-AA
#   gray dotted horizontal line = random-archetype expectation (z = 0)
# ============================================================

from matplotlib.lines import Line2D

# ------------------------------------------------------------
# Visual conventions
# ------------------------------------------------------------
CONDITIONS = ["intact", "word"]

COND_COLORS = {
    "intact": "#7B3294",   # purple
    "word":   "#2CA25F",   # green
}

COND_LABELS = {
    "intact": "Intact narrative",
    "word":   "Word-scrambled",
}

ANALYSIS_STYLES = {
    "spatial": {
        "linestyle": "-",
        "label": "Spatial MS-AA",
    },
    "temporal": {
        "linestyle": "--",
        "label": "Temporal MS-AA",
    },
}


# ============================================================
# FIGURE
# ============================================================
fig, axes = plt.subplots(
    1,
    2,
    figsize=(11.5, 4.9),
    sharex=True,
)

panel_info = [
    {
        "ax": axes[0],
        "metric": "gini",
        "title": "Localization relative to random archetypes",
        "ylabel": "Observed − random (z)",
        "letter": "A",
    },
    {
        "ax": axes[1],
        "metric": "participation_ratio",
        "title": "Spatial extent relative to random archetypes",
        "ylabel": "Observed − random (z)",
        "letter": "B",
    },
]


# ============================================================
# PLOT EACH NULL-CALIBRATED METRIC
# ============================================================
for info in panel_info:

    ax = info["ax"]

    sub = (
        null_df[
            null_df["metric"].astype(str) == info["metric"]
        ]
        .copy()
    )

    for condition in CONDITIONS:

        for analysis_type in ["spatial", "temporal"]:

            cur = (
                sub[
                    (
                        sub["condition"].astype(str)
                        == condition
                    )
                    & (
                        sub["analysis_type"].astype(str)
                        == analysis_type
                    )
                ]
                .sort_values("component_ratio")
            )

            if len(cur) == 0:
                continue

            ax.plot(
                cur["component_ratio"],
                cur["z"],
                color=COND_COLORS[condition],
                linestyle=ANALYSIS_STYLES[
                    analysis_type
                ]["linestyle"],
                marker="o",
                markersize=4.0,
                linewidth=2.0,
                alpha=0.95,
            )

    # Random-archetype expectation
    ax.axhline(
        0,
        color="0.55",
        linestyle=":",
        linewidth=1.3,
        zorder=0,
    )

    ax.set_title(
        info["title"],
        fontsize=13.5,
        fontweight="bold",
        pad=10,
    )

    ax.set_xlabel(
        "Normalized representational ratio",
        fontsize=11.5,
    )

    ax.set_ylabel(
        info["ylabel"],
        fontsize=11.5,
    )

    ax.tick_params(
        axis="both",
        labelsize=10,
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # Panel letter
    ax.text(
        -0.10,
        1.05,
        info["letter"],
        transform=ax.transAxes,
        fontsize=17,
        fontweight="bold",
        ha="left",
        va="top",
    )


# ============================================================
# LEGENDS
#
# Separate color and linestyle semantics so the reader does not
# need to infer what each visual channel means.
# ============================================================

condition_handles = [
    Line2D(
        [0],
        [0],
        color=COND_COLORS["intact"],
        linewidth=2.5,
        label="Intact narrative",
    ),
    Line2D(
        [0],
        [0],
        color=COND_COLORS["word"],
        linewidth=2.5,
        label="Word-scrambled",
    ),
]

analysis_handles = [
    Line2D(
        [0],
        [0],
        color="0.25",
        linewidth=2.0,
        linestyle="-",
        label="Spatial MS-AA",
    ),
    Line2D(
        [0],
        [0],
        color="0.25",
        linewidth=2.0,
        linestyle="--",
        label="Temporal MS-AA",
    ),
    Line2D(
        [0],
        [0],
        color="0.55",
        linewidth=1.3,
        linestyle=":",
        label="Random-archetype expectation",
    ),
]


# Condition legend
legend1 = fig.legend(
    handles=condition_handles,
    title="Condition",
    loc="lower center",
    bbox_to_anchor=(0.30, -0.02),
    ncol=2,
    frameon=False,
    fontsize=10,
    title_fontsize=10,
)

# Analysis-type legend
legend2 = fig.legend(
    handles=analysis_handles,
    title="Analysis",
    loc="lower center",
    bbox_to_anchor=(0.73, -0.02),
    ncol=3,
    frameon=False,
    fontsize=10,
    title_fontsize=10,
)


# ============================================================
# FINAL FORMATTING
# ============================================================

fig.suptitle(
    "Network localization relative to random-archetype nulls",
    fontsize=16,
    fontweight="bold",
    y=0.98,
)

fig.subplots_adjust(
    left=0.09,
    right=0.98,
    top=0.83,
    bottom=0.22,
    wspace=0.28,
)

fig.savefig(
    FIG_PNG,
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    FIG_PDF,
    bbox_inches="tight",
)

plt.show()

print("Saved:", FIG_PNG)
print("Saved:", FIG_PDF)